# 🗺️ Buscas Informadas — Animação Interativa
## Visualizando A\*, Busca Gulosa e Custo Uniforme passo a passo
### Inteligência Artificial — Ciência da Computação

---

Este notebook apresenta as **Buscas Informadas (Heurísticas)** de forma **visual e interativa**.
Use os controles deslizantes de cada seção para avançar passo a passo e observar o comportamento dos algoritmos.

**Conteúdo:**

| Seção | Descrição |
|-------|-----------|
| 🧩 **Parte 1 — Labirinto** | A\* navegando em um labirinto gerado automaticamente |
| 🏙️ **Parte 2 — Cidades Brasileiras** | Comparação interativa de A\*, Gulosa e Custo Uniforme |
| 📊 **Parte 3 — Análise e Exercícios** | Tabela comparativa e desafios para o aluno |

---

### Legenda de Cores (válida para todos os exemplos)

| Cor | Significado |
|-----|-------------|
| ⬛ Preto | Parede / Obstáculo |
| ⬜ Branco | Caminho livre não visitado |
| 🟦 Azul | Nó explorado (lista fechada) |
| 🟨 Amarelo | Fronteira (lista aberta) |
| 🟧 Laranja | Nó sendo expandido agora |
| 🟩 Verde | Caminho final encontrado |
| 🔴 Vermelho | Ponto de partida (S) |
| 🟣 Roxo | Objetivo (G) |


In [ ]:
# Bibliotecas

!pip install numpy matplotlib networkx ipywidgets

In [ ]:
# =====================================================
# IMPORTAÇÕES — execute esta célula primeiro
# =====================================================
# Se estiver no Google Colab, descomente a linha abaixo:
# !pip install ipywidgets networkx matplotlib --quiet

import sys, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import heapq
import networkx as nx
import ipywidgets as widgets
from IPython.display import display, clear_output

warnings.filterwarnings('ignore')
sys.setrecursionlimit(10000)

print('✅ Bibliotecas carregadas: numpy, matplotlib, networkx, ipywidgets')


---
## 🧩 Parte 1 — Labirinto com A\*

### O problema

Um robô precisa encontrar o caminho da entrada (🔴 **S**) até a saída (🟣 **G**) de um labirinto.

- A **Busca Cega (BFS/DFS)** explora sem direção, podendo visitar muitos caminhos inúteis.
- O **A\*** usa uma **heurística** — a *Distância de Manhattan* — para priorizar células mais próximas do objetivo.

### Como o A\* funciona no labirinto?

A cada passo o algoritmo mantém duas listas:

| Lista | O que contém | Cor |
|-------|-------------|-----|
| **Aberta (Fronteira)** | Células candidatas a explorar, ordenadas por `f(n)` | 🟨 Amarelo |
| **Fechada (Explorados)** | Células já analisadas | 🟦 Azul |

O nó com **menor** `f(n) = g(n) + h(n)` é expandido primeiro:
- `g(n)` = número de passos percorridos desde o início
- `h(n)` = Distância de Manhattan até o objetivo

### Distância de Manhattan

```
h(linha, coluna) = |linha_atual − linha_objetivo| + |coluna_atual − coluna_objetivo|
```

> 📌 Use o **controle deslizante** abaixo para ver o A\* explorando o labirinto passo a passo!


In [ ]:
# =====================================================
# PARTE 1: LABIRINTO COM A* — ANIMAÇÃO INTERATIVA
# =====================================================

# -- 1.1  Geração do labirinto (Recursive Backtracking / DFS) --
def gerar_labirinto(linhas, colunas, semente=42):
    # Gera um labirinto perfeito (sem ciclos, sempre resolúvel) usando DFS.
    # O labirinto físico tem (2*linhas+1) x (2*colunas+1) células.
    rng = np.random.default_rng(semente)
    maze = np.ones((linhas * 2 + 1, colunas * 2 + 1), dtype=int)

    def escavar(r, c):
        maze[r * 2 + 1, c * 2 + 1] = 0
        dirs = [(0, 1), (0, -1), (1, 0), (-1, 0)]
        rng.shuffle(dirs)
        for dr, dc in dirs:
            nr, nc = r + dr, c + dc
            if 0 <= nr < linhas and 0 <= nc < colunas:
                if maze[nr * 2 + 1, nc * 2 + 1] == 1:
                    maze[r * 2 + 1 + dr, c * 2 + 1 + dc] = 0
                    escavar(nr, nc)

    escavar(0, 0)
    return maze


# Gera o labirinto (9 x 13 células lógicas -> 19 x 27 físicas)
L_LOG, C_LOG = 9, 13
labirinto = gerar_labirinto(L_LOG, C_LOG, semente=42)
L, C = labirinto.shape
INICIO_L   = (1, 1)
OBJETIVO_L = (L - 2, C - 2)

print(f'Labirinto: {L}x{C} células físicas  ({L_LOG}x{C_LOG} lógicas)')
print(f'Início  : {INICIO_L}')
print(f'Objetivo: {OBJETIVO_L}')


# -- 1.2  Heurística de Manhattan --
def manhattan(pos, objetivo):
    return abs(pos[0] - objetivo[0]) + abs(pos[1] - objetivo[1])


# -- 1.3  A* com registro de passos --
def a_estrela_labirinto(maze, inicio, objetivo):
    # Executa A* e salva um snapshot a cada expansão para animação.
    rows, cols = maze.shape
    fila   = [(manhattan(inicio, objetivo), 0, inicio, [inicio])]
    g_min  = {inicio: 0}
    aberta = {inicio}
    fechada = set()
    passos  = []

    while fila:
        f, g, pos, caminho = heapq.heappop(fila)
        if pos in fechada:
            continue
        aberta.discard(pos)
        fechada.add(pos)

        passos.append({
            'aberta'   : frozenset(aberta),
            'fechada'  : frozenset(fechada),
            'atual'    : pos,
            'caminho'  : list(caminho),
            'g'        : g,
            'h'        : manhattan(pos, objetivo),
            'f'        : f,
            'encontrou': pos == objetivo,
        })

        if pos == objetivo:
            return passos, caminho

        r, c = pos
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and maze[nr, nc] == 0:
                viz = (nr, nc)
                ng  = g + 1
                if viz not in g_min or g_min[viz] > ng:
                    g_min[viz] = ng
                    heapq.heappush(
                        fila,
                        (ng + manhattan(viz, objetivo), ng, viz, caminho + [viz])
                    )
                    if viz not in fechada:
                        aberta.add(viz)

    return passos, None


passos_lab, caminho_lab = a_estrela_labirinto(labirinto, INICIO_L, OBJETIVO_L)

print(f'\nA* concluído!')
print(f'  Passos de expansão : {len(passos_lab)}')
print(f'  Nós explorados     : {len(passos_lab[-1]["fechada"])}')
if caminho_lab:
    print(f'  Comprimento do caminho: {len(caminho_lab)} células')
else:
    print('  Caminho não encontrado!')


# -- 1.4  Visualização --
# Grade de cores:
#   0=parede  1=livre  2=explorado  3=fronteira
#   4=atual   5=caminho  6=início  7=objetivo
_CMAP_LAB = ListedColormap([
    '#1a1a1a',  # 0 parede
    '#f8f8f8',  # 1 livre
    '#74b9ff',  # 2 explorado (azul)
    '#fdcb6e',  # 3 fronteira (amarelo)
    '#e17055',  # 4 atual     (laranja)
    '#00b894',  # 5 caminho   (verde)
    '#d63031',  # 6 início    (vermelho)
    '#6c5ce7',  # 7 objetivo  (roxo)
])


def _grid_lab(maze, passo, inicio, objetivo):
    grid = np.where(maze == 0, 1, 0).astype(float)
    for r, c in passo['fechada']:
        grid[r, c] = 2
    for r, c in passo['aberta']:
        grid[r, c] = 3
    if passo['encontrou']:
        for r, c in passo['caminho']:
            grid[r, c] = 5
    grid[passo['atual'][0], passo['atual'][1]] = 4
    grid[inicio[0],   inicio[1]]   = 6
    grid[objetivo[0], objetivo[1]] = 7
    return grid


def desenhar_passo_lab(num_passo):
    passo = passos_lab[num_passo]
    grid  = _grid_lab(labirinto, passo, INICIO_L, OBJETIVO_L)

    fig, ax = plt.subplots(figsize=(13, 8))
    ax.imshow(grid, cmap=_CMAP_LAB, vmin=0, vmax=7, aspect='equal')
    ax.set_xticks(np.arange(-0.5, C, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, L, 1), minor=True)
    ax.grid(which='minor', color='#aaaaaa', linewidth=0.25)
    ax.tick_params(which='both', bottom=False, left=False,
                   labelbottom=False, labelleft=False)
    ax.text(INICIO_L[1],   INICIO_L[0],   'S', ha='center', va='center',
            fontsize=8, fontweight='bold', color='white')
    ax.text(OBJETIVO_L[1], OBJETIVO_L[0], 'G', ha='center', va='center',
            fontsize=8, fontweight='bold', color='white')

    status = '✅ OBJETIVO ENCONTRADO!' if passo['encontrou'] else '🔍 Explorando…'
    ax.set_title(
        f"A* no Labirinto — Passo {num_passo + 1}/{len(passos_lab)}   |   {status}\n"
        f"Nó atual: {passo['atual']}   "
        f"g(n)={passo['g']}   h(n)={passo['h']}   f(n)={passo['f']}   "
        f"Explorados: {len(passo['fechada'])}   Fronteira: {len(passo['aberta'])}",
        fontsize=9, pad=8,
    )
    legendas = [
        mpatches.Patch(color='#1a1a1a', label='Parede'),
        mpatches.Patch(color='#f8f8f8', label='Livre'),
        mpatches.Patch(color='#74b9ff', label='Explorado'),
        mpatches.Patch(color='#fdcb6e', label='Fronteira'),
        mpatches.Patch(color='#e17055', label='Nó Atual'),
        mpatches.Patch(color='#00b894', label='Caminho Final'),
        mpatches.Patch(color='#d63031', label='Início (S)'),
        mpatches.Patch(color='#6c5ce7', label='Objetivo (G)'),
    ]
    ax.legend(handles=legendas, loc='upper left',
              bbox_to_anchor=(1.01, 1), fontsize=8,
              framealpha=0.9, title='Legenda')
    plt.tight_layout()
    plt.show()


# -- 1.5  Widget Interativo --
print('\n' + '=' * 60)
print('CONTROLE INTERATIVO — LABIRINTO COM A*')
print('=' * 60)
print(f'Total de passos: {len(passos_lab)}')

_sl_lab    = widgets.IntSlider(
    value=0, min=0, max=len(passos_lab) - 1, step=1,
    description='Passo:', continuous_update=False,
    style={'description_width': '60px'},
    layout=widgets.Layout(width='85%'),
)
_btn_ini_l = widgets.Button(description='⏮ Início',   button_style='warning', layout=widgets.Layout(width='100px'))
_btn_ant_l = widgets.Button(description='◀ Anterior', button_style='info',    layout=widgets.Layout(width='120px'))
_btn_pro_l = widgets.Button(description='Próximo ▶',  button_style='success', layout=widgets.Layout(width='120px'))
_btn_fim_l = widgets.Button(description='Fim ⏭',      button_style='warning', layout=widgets.Layout(width='100px'))
_out_lab   = widgets.Output()


def _atualizar_lab(p):
    with _out_lab:
        clear_output(wait=True)
        desenhar_passo_lab(p)


_sl_lab.observe(lambda c: _atualizar_lab(_sl_lab.value), names='value')
_btn_ini_l.on_click(lambda b: setattr(_sl_lab, 'value', 0))
_btn_fim_l.on_click(lambda b: setattr(_sl_lab, 'value', _sl_lab.max))
_btn_ant_l.on_click(lambda b: setattr(_sl_lab, 'value', max(0, _sl_lab.value - 1)))
_btn_pro_l.on_click(lambda b: setattr(_sl_lab, 'value', min(_sl_lab.max, _sl_lab.value + 1)))

display(
    widgets.VBox([
        _sl_lab,
        widgets.HBox([_btn_ini_l, _btn_ant_l, _btn_pro_l, _btn_fim_l]),
    ]),
    _out_lab,
)
_atualizar_lab(0)


---
## 🏙️ Parte 2 — Grafo de Cidades Brasileiras

### O problema de rota

Queremos encontrar o **melhor caminho** entre duas cidades conectadas por estradas.
As distâncias (em km) são os pesos das arestas; a heurística é a **distância em linha reta** até o destino.

### Os três algoritmos comparados

| Algoritmo | Ordena por | Garante solução ótima? | Característica |
|-----------|-----------|------------------------|---------------|
| **Custo Uniforme (UCS)** | `g(n)` = custo real acumulado | ✅ Sim | Explora em todas as direções, garante o menor custo |
| **Busca Gulosa** | `h(n)` = estimativa até o destino | ❌ Não | Rápida, mas pode errar o caminho ótimo |
| **A\*** | `f(n) = g(n) + h(n)` | ✅ Sim (com heurística admissível/consistente) | Melhor equilíbrio entre velocidade e otimidade |

### Heurística: distância euclidiana escalada

As posições das cidades são aproximadas num plano 2D. A heurística é:
```
h(cidade) = distância_euclidiana(cidade, destino) × escala
```
Para garantir **admissibilidade** (h(n) ≤ custo real), o fator de escala é calculado automaticamente como o **menor quociente** entre a distância rodoviária e a distância euclidiana entre todos os pares de cidades vizinhas no grafo. Assim, a heurística nunca superestima o custo real e o A\* encontra o caminho ótimo.

> 📌 Selecione **Origem**, **Destino** e **Algoritmo**, clique em **▶ Executar Busca** e use o controle deslizante para avançar passo a passo!


In [ ]:
# =====================================================
# PARTE 2: GRAFO DE CIDADES — ANIMAÇÃO INTERATIVA
# =====================================================

# -- 2.1  Definição do grafo --
# Posições visuais aproximadas em plano 2D (escala relativa)
CIDADES_POS = {
    'Manaus'         : (1.0,  9.5),
    'Belém'          : (5.5,  9.0),
    'Fortaleza'      : (8.5,  8.2),
    'Recife'         : (9.5,  6.5),
    'Salvador'       : (8.5,  5.0),
    'Brasília'       : (6.5,  5.5),
    'Goiânia'        : (5.8,  4.5),
    'Belo Horizonte' : (7.5,  3.8),
    'Rio de Janeiro' : (8.2,  2.8),
    'São Paulo'      : (7.0,  2.4),
    'Curitiba'       : (6.4,  1.5),
    'Porto Alegre'   : (5.9,  0.4),
    'Campo Grande'   : (5.0,  3.4),
    'Cuiabá'         : (4.0,  5.0),
    'Porto Velho'    : (2.0,  7.0),
}

# (cidade1, cidade2, distância_rodoviária_km)
ARESTAS_CIDADES = [
    ('Manaus',         'Porto Velho',          900),
    ('Manaus',         'Belém',               1300),
    ('Porto Velho',    'Cuiabá',               900),
    ('Porto Velho',    'Belém',               1800),
    ('Belém',          'Fortaleza',            1500),
    ('Belém',          'Brasília',             2100),
    ('Fortaleza',      'Recife',                800),
    ('Fortaleza',      'Salvador',             1200),
    ('Recife',         'Salvador',              800),
    ('Salvador',       'Brasília',             1400),
    ('Salvador',       'Belo Horizonte',       1200),
    ('Brasília',       'Goiânia',               200),
    ('Brasília',       'Belo Horizonte',        750),
    ('Brasília',       'Cuiabá',              1100),
    ('Goiânia',        'Cuiabá',               900),
    ('Goiânia',        'Campo Grande',          900),
    ('Goiânia',        'São Paulo',            1000),
    ('Belo Horizonte', 'Rio de Janeiro',        450),
    ('Belo Horizonte', 'São Paulo',             600),
    ('Rio de Janeiro', 'São Paulo',             430),
    ('São Paulo',      'Curitiba',              410),
    ('São Paulo',      'Campo Grande',         1000),
    ('Curitiba',       'Porto Alegre',          700),
    ('Campo Grande',   'Cuiabá',               700),
]

# Dicionário de adjacência
grafo_cidades = {c: [] for c in CIDADES_POS}
for _c1, _c2, _d in ARESTAS_CIDADES:
    grafo_cidades[_c1].append((_c2, _d))
    grafo_cidades[_c2].append((_c1, _d))

# NetworkX para desenho
G_NX = nx.Graph()
for _cid in CIDADES_POS:
    G_NX.add_node(_cid, pos=CIDADES_POS[_cid])
for _c1, _c2, _d in ARESTAS_CIDADES:
    G_NX.add_edge(_c1, _c2, weight=_d)

print('✅ Grafo de cidades definido!')
print(f'   Cidades  : {len(CIDADES_POS)}')
print(f'   Conexões : {len(ARESTAS_CIDADES)}')


# -- 2.2  Heurística: distância euclidiana escalada --
# _ESCALA é o menor quociente (custo_aresta / distância_euclidiana) entre todos
# os pares vizinhos do grafo.  Isso garante que h(n) <= custo real para toda
# aresta, tornando a heurística admissível e o A* ótimo.
_ESCALA = min(
    dist / ((CIDADES_POS[c1][0] - CIDADES_POS[c2][0]) ** 2
            + (CIDADES_POS[c1][1] - CIDADES_POS[c2][1]) ** 2) ** 0.5
    for c1, c2, dist in ARESTAS_CIDADES
) * 0.9999  # margem de segurança para aritmética de ponto flutuante

def heur_eucl(cidade, objetivo):
    x1, y1 = CIDADES_POS[cidade]
    x2, y2 = CIDADES_POS[objetivo]
    return ((x1 - x2) ** 2 + (y1 - y2) ** 2) ** 0.5 * _ESCALA


# -- 2.3  Busca genérica com registro de passos --
def busca_cidades(grafo, inicio, objetivo, modo='astar'):
    # modo: 'astar' -> f=g+h | 'gulosa' -> f=h | 'ucs' -> f=g
    def pri(g_real, no):
        h = heur_eucl(no, objetivo)
        if modo == 'astar':
            return g_real + h
        if modo == 'gulosa':
            return h
        return g_real

    fila    = [(pri(0, inicio), 0, inicio, [inicio])]
    g_min   = {inicio: 0}
    aberta  = {inicio}
    fechada = set()
    passos  = []

    while fila:
        p, g_real, no, caminho = heapq.heappop(fila)
        if no in fechada:
            continue
        aberta.discard(no)
        fechada.add(no)

        passos.append({
            'aberta'   : frozenset(aberta),
            'fechada'  : frozenset(fechada),
            'atual'    : no,
            'caminho'  : list(caminho),
            'g'        : g_real,
            'h'        : heur_eucl(no, objetivo),
            'f'        : p,
            'encontrou': no == objetivo,
        })

        if no == objetivo:
            return passos, caminho

        for viz, custo in grafo[no]:
            ng = g_real + custo
            if viz not in g_min or g_min[viz] > ng:
                g_min[viz] = ng
                heapq.heappush(fila, (pri(ng, viz), ng, viz, caminho + [viz]))
                if viz not in fechada:
                    aberta.add(viz)

    return passos, None


# -- 2.4  Visualização do grafo de cidades --
def desenhar_passo_cidades(passo, inicio, objetivo, titulo=''):
    fechada = passo['fechada']
    aberta  = passo['aberta']
    atual   = passo['atual']
    caminho = passo['caminho']
    cam_set = set(zip(caminho, caminho[1:])) | set(zip(caminho[1:], caminho))

    cor_nos, tam_nos = [], []
    for cidade in G_NX.nodes():
        if cidade == objetivo:
            cor_nos.append('#6c5ce7'); tam_nos.append(900)
        elif cidade == inicio:
            cor_nos.append('#d63031'); tam_nos.append(900)
        elif cidade == atual:
            cor_nos.append('#e17055'); tam_nos.append(750)
        elif cidade in caminho:
            cor_nos.append('#00b894'); tam_nos.append(650)
        elif cidade in fechada:
            cor_nos.append('#74b9ff'); tam_nos.append(550)
        elif cidade in aberta:
            cor_nos.append('#fdcb6e'); tam_nos.append(550)
        else:
            cor_nos.append('#dfe6e9'); tam_nos.append(500)

    cor_ar, larg_ar = [], []
    for u, v in G_NX.edges():
        if (u, v) in cam_set or (v, u) in cam_set:
            cor_ar.append('#00b894'); larg_ar.append(3.5)
        else:
            cor_ar.append('#b2bec3'); larg_ar.append(1.0)

    fig, ax = plt.subplots(figsize=(15, 9))
    nx.draw_networkx_edges(G_NX, CIDADES_POS, ax=ax,
                           edge_color=cor_ar, width=larg_ar, alpha=0.8)
    pesos = nx.get_edge_attributes(G_NX, 'weight')
    nx.draw_networkx_edge_labels(G_NX, CIDADES_POS, pesos, ax=ax,
                                 font_size=6, font_color='#636e72',
                                 bbox=dict(boxstyle='round,pad=0.1',
                                           fc='white', alpha=0.7))
    nx.draw_networkx_nodes(G_NX, CIDADES_POS, ax=ax,
                           node_color=cor_nos, node_size=tam_nos, alpha=0.95)
    nx.draw_networkx_labels(G_NX, CIDADES_POS, ax=ax,
                            font_size=7, font_weight='bold')

    info = (
        f"Nó atual : {passo['atual']}\n"
        f"g(n) = {passo['g']:.0f} km\n"
        f"h(n) = {passo['h']:.0f} km\n"
        f"f(n) = {passo['f']:.0f} km\n"
        f"Explorados: {len(fechada)}  |  Fronteira: {len(aberta)}"
    )
    ax.text(0.01, 0.99, info, transform=ax.transAxes,
            fontsize=8, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.85))

    legendas = [
        mpatches.Patch(color='#d63031', label=f'Início ({inicio})'),
        mpatches.Patch(color='#6c5ce7', label=f'Objetivo ({objetivo})'),
        mpatches.Patch(color='#e17055', label='Nó Atual'),
        mpatches.Patch(color='#00b894', label='No Caminho'),
        mpatches.Patch(color='#74b9ff', label='Explorado'),
        mpatches.Patch(color='#fdcb6e', label='Fronteira'),
        mpatches.Patch(color='#dfe6e9', label='Não Visitado'),
    ]
    ax.legend(handles=legendas, loc='lower left',
              fontsize=8, framealpha=0.9, title='Legenda')

    status = '✅ CHEGOU!' if passo['encontrou'] else '🔍 Buscando…'
    ax.set_title(f'{titulo}   |   {status}', fontsize=10, pad=10)
    ax.axis('off')
    plt.tight_layout()
    plt.show()


# -- 2.5  Widget Interativo --
print('\n' + '=' * 60)
print('CONTROLE INTERATIVO — GRAFO DE CIDADES')
print('=' * 60)

_lista_cid = sorted(CIDADES_POS.keys())

_sel_ini = widgets.Dropdown(
    options=_lista_cid, value='São Paulo',
    description='Origem:', style={'description_width': '70px'},
    layout=widgets.Layout(width='210px'),
)
_sel_obj = widgets.Dropdown(
    options=_lista_cid, value='Fortaleza',
    description='Destino:', style={'description_width': '70px'},
    layout=widgets.Layout(width='210px'),
)
_sel_alg = widgets.RadioButtons(
    options=[('A*  (usa g+h; ótimo se h admissível/consistente)', 'astar'),
             ('Busca Gulosa  (usa só h)',            'gulosa'),
             ('Custo Uniforme  (usa só g, ótimo)',   'ucs')],
    value='astar',
    description='Algoritmo:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='310px'),
)
_btn_exec  = widgets.Button(
    description='▶ Executar Busca', button_style='primary',
    layout=widgets.Layout(width='160px'),
)
_sl_cid    = widgets.IntSlider(
    value=0, min=0, max=0, step=1,
    description='Passo:', continuous_update=False,
    style={'description_width': '60px'},
    layout=widgets.Layout(width='85%'),
)
_btn_ini_c = widgets.Button(description='⏮ Início',   button_style='warning', layout=widgets.Layout(width='100px'))
_btn_ant_c = widgets.Button(description='◀ Anterior', button_style='info',    layout=widgets.Layout(width='120px'))
_btn_pro_c = widgets.Button(description='Próximo ▶',  button_style='success', layout=widgets.Layout(width='120px'))
_btn_fim_c = widgets.Button(description='Fim ⏭',      button_style='warning', layout=widgets.Layout(width='100px'))
_out_cid   = widgets.Output()

_NOMES = {'astar': 'A*', 'gulosa': 'Busca Gulosa', 'ucs': 'Custo Uniforme'}
_est_c = {'passos': [], 'inicio': None, 'objetivo': None, 'modo': None}


def _exec_busca(b):
    ini  = _sel_ini.value
    obj  = _sel_obj.value
    modo = _sel_alg.value
    with _out_cid:
        clear_output(wait=True)
        if ini == obj:
            print('⚠️  Origem e Destino não podem ser iguais!')
            return
        passos, caminho = busca_cidades(grafo_cidades, ini, obj, modo=modo)
        _est_c.update({'passos': passos, 'inicio': ini,
                        'objetivo': obj, 'modo': modo})
        _sl_cid.max   = len(passos) - 1
        _sl_cid.value = 0
        print(f'✅ {_NOMES[modo]}: {ini} → {obj}')
        print(f'   Passos registrados : {len(passos)}')
        if caminho:
            print(f'   Caminho  : {" → ".join(caminho)}')
            print(f'   Custo    : {passos[-1]["g"]:.0f} km')
        else:
            print('   Caminho não encontrado!')
        print('\nUse o controle deslizante para ver passo a passo ▼')


def _atualizar_cid(p):
    if not _est_c['passos']:
        return
    n = len(_est_c['passos'])
    t = f"{_NOMES[_est_c['modo']]}: {_est_c['inicio']} → {_est_c['objetivo']}  |  Passo {p + 1}/{n}"
    with _out_cid:
        clear_output(wait=True)
        desenhar_passo_cidades(_est_c['passos'][p], _est_c['inicio'],
                                _est_c['objetivo'], t)


_btn_exec.on_click(_exec_busca)
_sl_cid.observe(lambda c: _atualizar_cid(_sl_cid.value), names='value')
_btn_ini_c.on_click(lambda b: setattr(_sl_cid, 'value', 0))
_btn_fim_c.on_click(lambda b: setattr(_sl_cid, 'value', _sl_cid.max))
_btn_ant_c.on_click(lambda b: setattr(_sl_cid, 'value', max(0, _sl_cid.value - 1)))
_btn_pro_c.on_click(lambda b: setattr(_sl_cid, 'value', min(_sl_cid.max, _sl_cid.value + 1)))

display(
    widgets.VBox([
        widgets.HBox([_sel_ini, _sel_obj, _sel_alg, _btn_exec],
                     layout=widgets.Layout(align_items='center', flex_wrap='wrap')),
        _sl_cid,
        widgets.HBox([_btn_ini_c, _btn_ant_c, _btn_pro_c, _btn_fim_c]),
    ]),
    _out_cid,
)
print("\n👆 Selecione origem, destino e algoritmo, depois clique em '▶ Executar Busca'")


---
## 📊 Parte 3 — Comparação, Análise e Exercícios

### Tabela Resumo

| Algoritmo | f(n) usado | Ótimo? | Usa heurística? | Quando usar |
|-----------|-----------|--------|----------------|-------------|
| **Custo Uniforme** | `g(n)` | ✅ Sim | ❌ Não | Sem heurística disponível |
| **Busca Gulosa** | `h(n)` | ❌ Não | ✅ Sim | Quando velocidade > otimalidade |
| **A\*** | `g(n) + h(n)` | ✅ Sim (com h admissível/consistente) | ✅ Sim | Melhor equilíbrio geral |

### Condição de admissibilidade

O A\* garante o caminho **ótimo** se a heurística for **admissível**:
> **h(n) ≤ custo_real(n → objetivo)** para todo nó n

A distância em linha reta e a distância de Manhattan sempre são admissíveis, pois nunca **superestimam** o custo real.

### O que observar nas animações?

- **Custo Uniforme** expande uma "bolha" em todas as direções a partir da origem.
- **Busca Gulosa** avança rapidamente em direção ao objetivo, mas pode tomar caminhos mais longos.
- **A\*** equilibra as duas tendências — geralmente explora menos nós que UCS e encontra melhor custo que a Gulosa.

---

### Exercícios

1. **Labirinto — Comparação de sementes**
   Altere `semente=42` para `semente=7` ou `semente=99` na função `gerar_labirinto`.
   Como o número de nós explorados pelo A\* muda? Por quê?

2. **Cidades — Busca Gulosa vs A\***
   Execute os dois algoritmos entre *Porto Alegre* e *Belém*.
   Qual encontra o menor custo em km? Qual explora menos nós?

3. **Heurística nula**
   Modifique `heur_eucl` para retornar sempre `0`.
   Qual algoritmo o A\* passa a se comportar? (Dica: observe a fórmula `f = g + 0`)

4. **Heurística inadmissível**
   Multiplique o retorno da heurística por `3` (superestima o custo).
   O A\* ainda encontra o caminho ótimo? Por quê?

5. **Novo destino**
   Adicione a cidade *Florianópolis* ao grafo com posição `(6.2, 1.0)` e conexões a:
   - *Curitiba*: 300 km
   - *Porto Alegre*: 480 km

   Execute o A\* de *Manaus* até *Florianópolis*.

6. **Pesquisa — IDA\***
   Procure sobre o algoritmo **IDA\*** (*Iterative Deepening A\**).
   Que problema de memória do A\* padrão ele resolve?
